# Intent Classifier v2 Training

This notebook trains the improved intent classifier with:
- **6 intents** (order_status, payment_info, product_price, product_stock, product_description, out_of_scope)
- **Early stopping** to prevent overfitting
- **Comprehensive evaluation** metrics

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset

# Project paths
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
MODEL_DIR = PROJECT_ROOT / "models" / "intent_classifier_v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Load Dataset

print("Loading dataset...")
print("=" * 60)

df = pd.read_csv(DATA_DIR / "intent_dataset_v2.csv")

# Clean data
df = df.dropna()
df = df[df['text'].str.len() > 0]

print(f"Total samples: {len(df)}")
print(f"\nIntent distribution:")
print(df['intent'].value_counts())

In [ ]:
# Cell 3: Create Label Mappings

# Define 6 intents
INTENTS = [
    "order_status",
    "payment_info",
    "product_price",
    "product_stock",
    "product_description",
    "out_of_scope"
]

# Create mappings
label2id = {intent: idx for idx, intent in enumerate(INTENTS)}
id2label = {idx: intent for idx, intent in enumerate(INTENTS)}

# Add label column
df['label'] = df['intent'].map(label2id)

print("Label mappings:")
print(f"  label2id: {label2id}")
print(f"  id2label: {id2label}")

# Verify all intents are mapped
assert df['label'].isna().sum() == 0, "Some intents couldn't be mapped!"
print(f"\nAll {len(df)} samples mapped successfully!")

In [ ]:
# Cell 4: Train/Val/Test Split

# Split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    df, 
    test_size=0.3, 
    random_state=42, 
    stratify=df['label']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['label']
)

print("Dataset splits:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")

# Verify stratification
print(f"\nTrain intent distribution:")
print(train_df['intent'].value_counts())

In [ ]:
# Cell 5: Load Tokenizer and Model

MODEL_NAME = "cahya/distilbert-base-indonesian"

print(f"Loading model: {MODEL_NAME}")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded!")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENTS),
    id2label=id2label,
    label2id=label2id
)
print(f"Model loaded with {len(INTENTS)} labels!")

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

In [ ]:
# Cell 6: Tokenize Datasets

MAX_LENGTH = 128

def tokenize_data(df, tokenizer, max_length=MAX_LENGTH):
    """Tokenize a dataframe into a HuggingFace Dataset"""
    # Reset index to avoid issues
    df = df.reset_index(drop=True)
    
    encodings = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None
    )
    
    # Ensure labels are integers
    labels = [int(x) for x in df['label'].tolist()]
    
    return Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })

print("Tokenizing datasets...")
train_dataset = tokenize_data(train_df, tokenizer)
val_dataset = tokenize_data(val_df, tokenizer)
test_dataset = tokenize_data(test_df, tokenizer)

print(f"Datasets tokenized!")
print(f"  Train: {len(train_dataset)}")
print(f"  Val: {len(val_dataset)}")
print(f"  Test: {len(test_dataset)}")

# Verify data types
print(f"\nSample data check:")
print(f"  input_ids type: {type(train_dataset[0]['input_ids'])}")
print(f"  labels type: {type(train_dataset[0]['labels'])}")
print(f"  labels value: {train_dataset[0]['labels']}")

In [ ]:
# Cell 7: Define Metrics

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    logits, labels = eval_pred
    
    # Ensure logits is a numpy array
    if hasattr(logits, 'numpy'):
        logits = logits.numpy()
    logits = np.array(logits)
    
    # Ensure labels is a numpy array  
    if hasattr(labels, 'numpy'):
        labels = labels.numpy()
    labels = np.array(labels)
    
    # Get predictions
    predictions = np.argmax(logits, axis=-1)
    
    # Calculate metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Metrics function defined!")

In [ ]:
# Cell 8: Training Configuration

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    
    # Training hyperparameters
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    warmup_steps=100,
    
    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    
    # Logging
    logging_dir=str(MODEL_DIR / "logs"),
    logging_steps=50,
    report_to="none",
    
    # Other settings
    seed=42,
    fp16=torch.cuda.is_available(),  # Use FP16 if GPU available
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  FP16: {training_args.fp16}")

In [ ]:
# Cell 9: Create Trainer

def preprocess_logits_for_metrics(logits, labels):
    """
    Preprocess logits before computing metrics.
    This handles cases where logits is a tuple (e.g., with hidden states).
    """
    # If logits is a tuple, take the first element (actual logits)
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Trainer created with early stopping (patience=3)!")

In [ ]:
# Cell 10: Train Model

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Train
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nTraining metrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 11: Save Best Model

BEST_MODEL_DIR = MODEL_DIR / "best_model"
BEST_MODEL_DIR.mkdir(exist_ok=True)

# Save model and tokenizer
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

# Save label mapping
label_mapping = {
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()},  # JSON needs string keys
    'intents': INTENTS,
    'confidence_threshold': 0.7  # Default threshold, will be tuned
}

with open(BEST_MODEL_DIR / "label_mapping.json", 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"Model saved to: {BEST_MODEL_DIR}")
print(f"Files saved:")
for f in BEST_MODEL_DIR.iterdir():
    print(f"  - {f.name}")

In [ ]:
# Cell 12: Evaluate on Test Set

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Get predictions
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

all_preds = []
all_labels = []
all_confidences = []

for item in test_dataset:
    input_ids = torch.tensor([item['input_ids']]).to(device)
    attention_mask = torch.tensor([item['attention_mask']]).to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred].item()
    
    all_preds.append(pred)
    all_labels.append(item['labels'])
    all_confidences.append(confidence)

# Calculate metrics
accuracy = np.mean([t == p for t, p in zip(all_labels, all_preds)])
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Cell 13: Classification Report

print("\nClassification Report:")
print("-" * 60)
print(classification_report(all_labels, all_preds, target_names=INTENTS, digits=4))

In [ ]:
# Cell 14: Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=INTENTS, yticklabels=INTENTS, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Intent Classifier v2 - Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=300)
plt.show()

print(f"Confusion matrix saved to: {MODEL_DIR / 'confusion_matrix.png'}")

In [ ]:
# Cell 15: Confidence Analysis

print("\nConfidence Score Analysis:")
print("-" * 60)

correct_mask = [t == p for t, p in zip(all_labels, all_preds)]
correct_conf = [c for c, m in zip(all_confidences, correct_mask) if m]
wrong_conf = [c for c, m in zip(all_confidences, correct_mask) if not m]

print(f"Correct predictions ({len(correct_conf)}):")
print(f"  Mean confidence: {np.mean(correct_conf):.4f}")
print(f"  Min confidence: {np.min(correct_conf):.4f}")
print(f"  Std confidence: {np.std(correct_conf):.4f}")

if len(wrong_conf) > 0:
    print(f"\nWrong predictions ({len(wrong_conf)}):")
    print(f"  Mean confidence: {np.mean(wrong_conf):.4f}")
    print(f"  Max confidence: {np.max(wrong_conf):.4f}")
    print(f"  Std confidence: {np.std(wrong_conf):.4f}")

# Confidence distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(correct_conf, bins=20, alpha=0.7, label='Correct', color='green')
if len(wrong_conf) > 0:
    axes[0].hist(wrong_conf, bins=20, alpha=0.7, label='Wrong', color='red')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence Distribution')
axes[0].legend()

data_to_plot = [correct_conf]
labels_to_plot = ['Correct']
if len(wrong_conf) > 0:
    data_to_plot.append(wrong_conf)
    labels_to_plot.append('Wrong')
axes[1].boxplot(data_to_plot, labels=labels_to_plot)
axes[1].set_ylabel('Confidence Score')
axes[1].set_title('Confidence by Correctness')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'confidence_analysis.png', dpi=300)
plt.show()

In [ ]:
# Cell 16: Save Test Results

# Create results dataframe
test_results = pd.DataFrame({
    'text': test_df['text'].tolist(),
    'true_label': [id2label[l] for l in all_labels],
    'predicted_label': [id2label[p] for p in all_preds],
    'confidence': all_confidences,
    'correct': correct_mask
})

test_results.to_csv(MODEL_DIR / 'test_results.csv', index=False)
print(f"Test results saved to: {MODEL_DIR / 'test_results.csv'}")

# Save training summary
training_summary = {
    'model_name': MODEL_NAME,
    'num_intents': len(INTENTS),
    'intents': INTENTS,
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_df),
    'test_accuracy': float(accuracy),
    'training_args': {
        'epochs': training_args.num_train_epochs,
        'learning_rate': training_args.learning_rate,
        'batch_size': training_args.per_device_train_batch_size,
    },
    'confidence_stats': {
        'correct_mean': float(np.mean(correct_conf)),
        'correct_min': float(np.min(correct_conf)),
        'wrong_mean': float(np.mean(wrong_conf)) if len(wrong_conf) > 0 else None,
        'wrong_max': float(np.max(wrong_conf)) if len(wrong_conf) > 0 else None,
    }
}

with open(MODEL_DIR / 'training_summary.json', 'w') as f:
    json.dump(training_summary, f, indent=2)
print(f"Training summary saved to: {MODEL_DIR / 'training_summary.json'}")

In [ ]:
# Cell 17: Quick Test

from transformers import pipeline

print("\n" + "=" * 60)
print("QUICK MODEL TEST")
print("=" * 60)

# Load model for inference
classifier = pipeline(
    "text-classification",
    model=str(BEST_MODEL_DIR),
    device=-1
)

# Test queries
test_queries = [
    "cek pesanan 12345",
    "bisa bayar pakai gopay?",
    "harga beanie berapa",
    "stok sepatu masih ada?",
    "spesifikasi laptop gimana",
    "mau refund dong",
    "brg gw mn bro",  # typo test
    "bs cod g",  # abbreviation test
]

print("\nTest predictions:")
print("-" * 60)

for query in test_queries:
    result = classifier(query)[0]
    print(f"\n\"{query}\"")
    print(f"  -> {result['label']} (confidence: {result['score']:.3f})")

In [ ]:
# Cell 18: Summary

print("\n" + "=" * 60)
print("TRAINING COMPLETE - SUMMARY")
print("=" * 60)

print(f"""
Model: {MODEL_NAME}
Intents: {len(INTENTS)} ({', '.join(INTENTS)})

Dataset:
  - Train: {len(train_df)}
  - Val: {len(val_df)}
  - Test: {len(test_df)}

Performance:
  - Test Accuracy: {accuracy:.2%}
  - Correct predictions mean confidence: {np.mean(correct_conf):.4f}

Files saved:
  - {BEST_MODEL_DIR}
  - {MODEL_DIR / 'test_results.csv'}
  - {MODEL_DIR / 'training_summary.json'}
  - {MODEL_DIR / 'confusion_matrix.png'}
  - {MODEL_DIR / 'confidence_analysis.png'}

Next steps:
  1. Run confidence threshold analysis (notebook 04)
  2. Compare with baseline v1 (notebook 05)
  3. Implement entity extraction (Phase 3)
""")